In [1]:
from dotenv import load_dotenv
load_dotenv()
from global_values import WORLD_STABLE_COIN
from db_manager_v2 import engine, execute_sell_percentage_investment
from db_manager_v2 import Base, Security, get_session, allocate_from_security_to_investment
from sqlalchemy import create_engine, text
from sqlalchemy.orm import sessionmaker
from dev_setup import full_dev_setup

2026-07-01 18:28:14 | INFO     | Logging initialized. Writing to logs/trades.log


In [2]:
# Full reset (development only)
with engine.connect() as conn:
    conn.execute(text("DROP SCHEMA public CASCADE;"))
    conn.execute(text("CREATE SCHEMA public;"))
    conn.commit()

# Recreate all tables

Base.metadata.create_all(engine)

print("✅ Database has been fully reset and tables recreated.")

✅ Database has been fully reset and tables recreated.


In [3]:


result = full_dev_setup(public_key="JCDgrYTDUiRPPyLCpkoishE7DHvLJVqadzDoAuQNsazF")

if result.is_ok:
    print("✅ Setup done!")
else:
    print("❌ Error:", result.error)


from db_manager_v2 import (
    sync_wallet_balances,
    allocate_from_security_to_investment,
    get_open_investments,
    get_wallet_summary
)
from sqlalchemy import select, text, func
import time

def fresh_wallet_setup(wallet_pk: int = 1):
    with get_session(engine) as session:
        print("=== Starting Fresh Wallet Setup ===")
        
        # 1. Sync on-chain balances into Assets
        print("\n[1/3] Syncing wallet balances...")
        sync_result = sync_wallet_balances(session=session, wallet_pk=wallet_pk)
        print(sync_result)
    
        # 2. Create Savings Securities from current Assets (simple & safe version)
        print("\n[2/3] Creating Savings Securities from Assets...")
        
        assets = session.execute(
            text("""
                SELECT a.id, a.coin, t.id as token_mint
                FROM asset_table a
                LEFT JOIN token_table t ON a.coin = t.id
                WHERE a.wallet = :wallet_pk
            """),
            {"wallet_pk": wallet_pk}
        ).fetchall()
    
        created = 0
        for asset in assets:
            asset_id = asset.id
            
            # Check if this asset already has an open Savings Security
            existing = session.execute(
                select(Security.id).where(
                    Security.parent_id == asset_id,
                    Security.isSavings == True,
                    Security.isClosed == False
                )
            ).scalar()
    
            if not existing:
                # Create new Savings Security
                new_savings = Security(
                    parent_id=asset_id,
                    amount=0,  # Will be updated by sync if needed
                    purchase_price_usdc=0.0,
                    purchase_time_ms=int(time.time() * 1000),
                    buy_fee_native_lamports=0,
                    buy_fee_usdc=0.0,
                    buy_tx_id=f"initial_savings_{asset_id}",
                    isClosed=False,
                    isSavings=True,
                    isGas=False,
                )
                session.add(new_savings)
                created += 1
    
        session.commit()
        print(f"Created {created} new Savings Securities")
    
        # 3. Final summary
        print("\n[3/3] Setup complete. Current state:")
        print(get_wallet_summary(session=session, wallet_pk=wallet_pk))
    
        return "Fresh setup finished successfully."


# ====================== RUN THIS ======================
fresh_wallet_setup(wallet_pk=1)

2026-07-01 18:28:16 | INFO     | [SETUP] Starting full development setup...
2026-07-01 18:28:16 | INFO     | [SETUP] Database tables created
2026-07-01 18:28:16 | INFO     | [SETUP] Default user 'adam' created
2026-07-01 18:28:16 | INFO     | HTTP Request: POST https://mainnet.helius-rpc.com/?api-key=c0dcc617-ab44-4343-8eb6-9cb7ca174243 "HTTP/1.1 200 OK"
2026-07-01 18:28:16 | INFO     | [RPC] Using PRIMARY: https://mainnet.helius-rpc.com/
2026-07-01 18:28:16 | INFO     | [SETUP] Added Platform Coin (So111111...)
2026-07-01 18:28:16 | INFO     | [SETUP] Added Stable Coin (EPjFWdd5...)
2026-07-01 18:28:16 | INFO     | HTTP Request: POST https://mainnet.helius-rpc.com/?api-key=c0dcc617-ab44-4343-8eb6-9cb7ca174243 "HTTP/1.1 200 OK"
2026-07-01 18:28:16 | INFO     | [0] Added token DtR4D9Ft (decimals=6)
2026-07-01 18:28:17 | INFO     | HTTP Request: POST https://mainnet.helius-rpc.com/?api-key=c0dcc617-ab44-4343-8eb6-9cb7ca174243 "HTTP/1.1 200 OK"
2026-07-01 18:28:17 | INFO     | [1] Added t

✅ Setup done!
=== Starting Fresh Wallet Setup ===

[1/3] Syncing wallet balances...


2026-07-01 18:28:27 | INFO     | HTTP Request: POST https://mainnet.helius-rpc.com/?api-key=c0dcc617-ab44-4343-8eb6-9cb7ca174243 "HTTP/1.1 200 OK"


Ok({'created_assets': 0, 'updated_assets': 0})

[2/3] Creating Savings Securities from Assets...
Created 14 new Savings Securities

[3/3] Setup complete. Current state:
Ok({'wallet_id': 1, 'open_investments_count': 0, 'total_investment_value_usdc': 0.0, 'total_tax_reserved_usdc': 0.0, 'total_gas_lamports': 0, 'total_gas_sol': 0.0, 'timestamp': 1782955707})


'Fresh setup finished successfully.'

In [4]:
from dev_setup import reconcile_onchain_to_securities


result = reconcile_onchain_to_securities(wallet_pk=1)   # ← change to your wallet id

if result.is_ok():
    print("✅ Reconciliation done!")
    print(result.ok_value)           # ← changed from result.value
else:
    print("❌ Error:", result.err_value)   # ← changed from result.error

2026-07-01 18:28:48 | INFO     | HTTP Request: POST https://mainnet.helius-rpc.com/?api-key=c0dcc617-ab44-4343-8eb6-9cb7ca174243 "HTTP/1.1 200 OK"
2026-07-01 18:28:49 | INFO     | HTTP Request: POST https://mainnet.helius-rpc.com/?api-key=c0dcc617-ab44-4343-8eb6-9cb7ca174243 "HTTP/1.1 200 OK"
2026-07-01 18:28:49 | INFO     | [RECONCILE] Reconciliation complete for wallet 1: {'assets_processed': 14, 'securities_created': 0, 'securities_updated': 4, 'unknown_tokens_added': 0, 'unknown_tokens_failed': 0}


✅ Reconciliation done!
{'assets_processed': 14, 'securities_created': 0, 'securities_updated': 4, 'unknown_tokens_added': 0, 'unknown_tokens_failed': 0}


In [5]:
from sqlalchemy import select
from db_manager_v2 import Security

with get_session(engine) as session:
    savings = session.execute(
        select(Security)
        .where(Security.isSavings == True, Security.isClosed == False)
    ).scalars().all()

    for s in savings:
        print(f"Security ID: {s.id} | Asset ID: {s.parent_id} | Amount: {s.amount}")

Security ID: 3 | Asset ID: 12 | Amount: 0
Security ID: 4 | Asset ID: 11 | Amount: 0
Security ID: 6 | Asset ID: 13 | Amount: 0
Security ID: 7 | Asset ID: 14 | Amount: 0
Security ID: 8 | Asset ID: 10 | Amount: 0
Security ID: 9 | Asset ID: 4 | Amount: 0
Security ID: 10 | Asset ID: 3 | Amount: 0
Security ID: 11 | Asset ID: 5 | Amount: 0
Security ID: 12 | Asset ID: 8 | Amount: 0
Security ID: 14 | Asset ID: 2 | Amount: 0
Security ID: 1 | Asset ID: 1 | Amount: 20553265
Security ID: 13 | Asset ID: 6 | Amount: 22863250
Security ID: 2 | Asset ID: 7 | Amount: 6818467
Security ID: 5 | Asset ID: 9 | Amount: 104


In [ ]:
from sqlalchemy import select, update
from db_manager_v2 import Security, Asset

with get_session(engine) as session:
    # Get all open Savings Securities
    savings_list = session.execute(
        select(Security).where(Security.isSavings == True, Security.isClosed == False)
    ).scalars().all()
    
    updated = 0
    for savings in savings_list:
        # Find the corresponding Asset
        asset = session.execute(
            select(Asset).where(Asset.id == savings.parent_id)
        ).scalar_one_or_none()
        
        if asset and asset.audited_amount_sum_lamports > 0:
            savings.amount = asset.audited_amount_sum_lamports
            updated += 1
    
    session.commit()
    print(f"Updated {updated} Savings Securities with real balances")

In [6]:
from db_manager_v2 import Security, Asset
from sqlalchemy import select
import time
with get_session(engine) as session:
    amount_to_gas = 20553265   # ← Change this (e.g. 0.2 SOL = 200_000_000)
    
    # Find an open Savings Security that has SOL (we'll take the first one with balance for simplicity)
    sol_savings = session.execute(
        select(Security)
        .join(Asset, Security.parent_id == Asset.id)
        .where(
            Security.isSavings == True,
            Security.isClosed == False,
            Asset.isNative == True,           # This helps identify SOL
            Security.amount > 0
        )
    ).scalars().first()
    
    if not sol_savings:
        print("No suitable SOL Savings Security found with balance.")
    else:
        if sol_savings.amount < amount_to_gas:
            print(f"Not enough SOL. Available: {sol_savings.amount}")
        else:
            sol_savings.amount -= amount_to_gas
    
            gas_sec = Security(
                parent_id=sol_savings.parent_id,
                amount=amount_to_gas,
                purchase_price_usdc=0.0,
                purchase_time_ms=int(time.time() * 1000),
                buy_fee_native_lamports=0,
                buy_fee_usdc=0.0,
                buy_tx_id=f"to_gas_{sol_savings.id}",
                isClosed=False,
                isSavings=False,
                isGas=True,
            )
            session.add(gas_sec)
            session.commit()

        print(f"✅ Moved {amount_to_gas} lamports SOL to Gas Security (id={gas_sec.id})")

✅ Moved 20553265 lamports SOL to Gas Security (id=15)


In [7]:
with get_session(engine) as session:
    # Example: Move from Security ID 3 (replace with the correct Savings Security ID)
    result = allocate_from_security_to_investment(
        session=session,
        security_id=2,              # ← Change to the Security ID you want to move from
        amount_lamports=6818467     # Amount to move to Investment
    )
    
    print(result)

Ok(1)


In [8]:
with get_session(engine) as session:
    # Example: Move from Security ID 3 (replace with the correct Savings Security ID)
    result = allocate_from_security_to_investment(
        session=session,
        security_id=5,              # ← Change to the Security ID you want to move from
        amount_lamports=104     # Amount to move to Investment
    )
    
    print(result)

Ok(2)


In [9]:
with get_session(engine) as session:
    # Example: Move from Security ID 3 (replace with the correct Savings Security ID)
    result = allocate_from_security_to_investment(
        session=session,
        security_id=13,              # ← Change to the Security ID you want to move from
        amount_lamports=22863250     # Amount to move to Investment
    )
    
    print(result)

Ok(3)


In [10]:
from sqlalchemy import select
from db_manager_v2 import Investment

with get_session(engine) as session:
    investments = session.execute(
        select(Investment).where(Investment.isClosed == False)
    ).scalars().all()
    
    print(f"Open Investments found: {len(investments)}")
    for inv in investments:
        print(f"ID: {inv.id} | Parent Asset: {inv.parent_id} | Amount: {inv.amount} | Buy TX: {inv.buy_tx_id}")

Open Investments found: 3
ID: 1 | Parent Asset: 7 | Amount: 6818467 | Buy TX: allocate_from_security_2
ID: 2 | Parent Asset: 9 | Amount: 104 | Buy TX: allocate_from_security_5
ID: 3 | Parent Asset: 6 | Amount: 22863250 | Buy TX: allocate_from_security_13


In [ ]:


with get_session(engine) as session:
    result = execute_sell_percentage_investment(
        session=session,
        investment_id=2,
        target_mint=WORLD_STABLE_COIN,
        sell_percentage=100.0,
        slippage_bps=310,
        estimated_gas_lamports=10_000 #200_000
    )
    
    print(result)
    session.expire_all()

# 2. Immediately verify inside the same session/transaction
    investments = session.execute(
        select(Investment)
    ).scalars().all()

    print(f"\nTotal investments in DB: {len(investments)}")
    for inv in investments:
        print(f"ID: {inv.id} | Amount: {inv.amount} | Closed: {inv.isClosed}")